<a href="https://colab.research.google.com/github/AthulBDev/Librarymanagement-python/blob/main/Pyprojectcode2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# 1. Pre-seed the root password to avoid interactive installation prompts
!echo "mysql-server mysql-server/root_password password root" | sudo debconf-set-selections
!echo "mysql-server mysql-server/root_password_again password root" | sudo debconf-set-selections

# 2. Update package lists and install MySQL Server
!sudo apt-get update
!sudo apt-get install -y mysql-server

# 3. Start the MySQL service
!sudo service mysql start

# 4. Configure root to use native password authentication with 'root' as the password
!sudo mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'root'; FLUSH PRIVILEGES;"

print("MySQL installed and configured successfully!")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [75.7 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,108 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,550 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]
Get:14 http

In [ ]:

!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 32.3 MB/s  0:00:00


SQL PASSWORD: root

ADMIN PASSWORD:   admin123

In [ ]:

import csv
import os
import pickle
import random
import mysql.connector

SQL_PASSWORD = ""

def get_database_connection():
    global SQL_PASSWORD
    if not SQL_PASSWORD:
        SQL_PASSWORD = input("Enter MySQL Password: ")
    try:
        con = mysql.connector.connect(
            host="localhost", user="root", password=SQL_PASSWORD
        )
        cursor = con.cursor()
        cursor.execute("CREATE DATABASE IF NOT EXISTS library_db")
        cursor.execute("USE library_db")
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS books (
                book_id INT PRIMARY KEY,
                title VARCHAR(100),
                author VARCHAR(100),
                status VARCHAR(20)
            )
        """)
        con.commit()
        return con
    except Exception as e:
        print("Database Connection Error:", e)
        return None

def add_book_sql():
    con = get_database_connection()
    if con is None:
        return
    cursor = con.cursor()
    print("\n--- ADD NEW BOOK (SQL) ---")
    try:
        b_id = int(input("Enter Book ID: "))
    except ValueError:
        print("Invalid Book ID! Please enter numbers only.")
        con.close()
        return
    title = input("Enter Book Title: ")
    author = input("Enter Author Name: ")
    status = "Available"
    query = "INSERT INTO books VALUES (%s, %s, %s, %s)"
    values = (b_id, title, author, status)
    try:
        cursor.execute(query, values)
        con.commit()
        print("Book successfully added to MySQL database!")
    except Exception as err:
        print("Error adding book:", err)
    finally:
        con.close()

def display_books_sql():
    con = get_database_connection()
    if con is None:
        return
    cursor = con.cursor()
    print("\n--- BOOK LIST (SQL DATABASE) ---")
    cursor.execute("SELECT * FROM books")
    records = cursor.fetchall()
    if len(records) == 0:
        print("No records found in the database.")
    else:
        print(f"{'ID':<10} {'Title':<30} {'Author':<25} {'Status':<15}")
        print("-" * 80)
        for row in records:
            print(f"{row[0]:<10} {row[1]:<30} {row[2]:<25} {row[3]:<15}")
    con.close()

def search_book_sql():
    con = get_database_connection()
    if con is None:
        return
    cursor = con.cursor()
    print("\n--- SEARCH BOOK (SQL) ---")
    try:
        b_id = int(input("Enter Book ID to search: "))
    except ValueError:
        print("Invalid Book ID!")
        con.close()
        return
    query = "SELECT * FROM books WHERE book_id = %s"
    cursor.execute(query, (b_id,))
    record = cursor.fetchone()
    if record:
        print("\nBook Found!")
        print(f"Book ID : {record[0]}")
        print(f"Title   : {record[1]}")
        print(f"Author  : {record[2]}")
        print(f"Status  : {record[3]}")
    else:
        print("Book not found in database.")
    con.close()

def check_book_status():
    con = get_database_connection()
    if con is None:
        return
    cursor = con.cursor()
    print("\n--- CHECK BOOK STATUS ---")
    try:
        b_id = int(input("Enter Book ID to check status: "))
    except ValueError:
        print("Invalid Book ID!")
        con.close()
        return
    query = "SELECT title, status FROM books WHERE book_id = %s"
    cursor.execute(query, (b_id,))
    record = cursor.fetchone()
    if record:
        print(f"\nBook Title : {record[0]}")
        print(f"Status     : {record[1]}")
        if record[1].lower() == "available":
            print("-> This book is currently IN THE LIBRARY.")
        else:
            print("-> This book is currently WITH A MEMBER (Issued).")
    else:
        print("Book ID not found in database.")
    con.close()

def update_book_status_manual():
    con = get_database_connection()
    if con is None:
        return
    cursor = con.cursor()
    print("\n--- UPDATE BOOK STATUS MANUALLY ---")
    try:
        b_id = int(input("Enter Book ID: "))
    except ValueError:
        print("Invalid Book ID! Please enter numbers only.")
        con.close()
        return
    cursor.execute("SELECT title, status FROM books WHERE book_id = %s", (b_id,))
    record = cursor.fetchone()
    if record:
        print(f"\nBook Title    : {record[0]}")
        print(f"Current Status: {record[1]}")
        print("\nSelect New Status:")
        print("1. Available (Returned to Library)")
        print("2. Issued (Taken by Member)")
        choice = input("Enter choice (1 or 2): ").strip()
        if choice == "1":
            new_status = "Available"
        elif choice == "2":
            new_status = "Issued"
        else:
            print("Invalid choice! Status update cancelled.")
            con.close()
            return
        cursor.execute("UPDATE books SET status = %s WHERE book_id = %s", (new_status, b_id))
        con.commit()
        print(f"Success! Book status updated to '{new_status}'.")
    else:
        print("Book ID not found in database.")
    con.close()

def delete_book_sql():
    con = get_database_connection()
    if con is None:
        return
    cursor = con.cursor()
    print("\n--- DELETE BOOK (SQL DATABASE) ---")
    try:
        b_id = int(input("Enter Book ID to delete: "))
    except ValueError:
        print("Invalid Book ID!")
        con.close()
        return
    try:
        cursor.execute("DELETE FROM books WHERE book_id = %s", (b_id,))
        con.commit()
        if cursor.rowcount > 0:
            print("Book deleted successfully from database!")
        else:
            print("Book ID not found.")
    except Exception as err:
        print("Error deleting book:", err)
    finally:
        con.close()

BINARY_FILE = "members.dat"

def add_member_binary():
    print("\n--- REGISTER MEMBER (BINARY FILE) ---")
    try:
        m_id = int(input("Enter Member ID: "))
    except ValueError:
        print("Invalid Member ID!")
        return
    m_name = input("Enter Member Name: ")
    m_phone = input("Enter Phone Number: ")
    member_data = [m_id, m_name, m_phone]
    f = open(BINARY_FILE, "ab")
    pickle.dump(member_data, f)
    f.close()
    print("Member details saved successfully in binary file!")

def display_members_binary():
    print("\n--- MEMBER LIST (BINARY FILE) ---")
    if not os.path.exists(BINARY_FILE):
        print("No member record binary file found.")
        return
    f = open(BINARY_FILE, "rb")
    print(f"{'Member ID':<12} {'Member Name':<25} {'Phone Number':<15}")
    print("-" * 55)
    try:
        while True:
            data = pickle.load(f)
            print(f"{data[0]:<12} {data[1]:<25} {data[2]:<15}")
    except EOFError:
        pass
    finally:
        f.close()

def delete_member_binary():
    print("\n--- DELETE MEMBER (BINARY FILE) ---")
    if not os.path.exists(BINARY_FILE):
        print("No member records found.")
        return
    try:
        m_id_to_delete = int(input("Enter Member ID to delete: "))
    except ValueError:
        print("Invalid Member ID!")
        return
    records = []
    found = False
    f = open(BINARY_FILE, "rb")
    try:
        while True:
            records.append(pickle.load(f))
    except EOFError:
        pass
    f.close()
    new_records = []
    for r in records:
        if r[0] == m_id_to_delete:
            found = True
        else:
            new_records.append(r)
    if found:
        f = open(BINARY_FILE, "wb")
        for r in new_records:
            pickle.dump(r, f)
        f.close()
        print("Member deleted successfully from binary file!")
    else:
        print("Member ID not found.")

CSV_FILE = "issues.csv"
TEXT_FILE = "library_notes.txt"

def issue_book_csv():
    print("\n--- ISSUE BOOK (CSV FILE) ---")
    issue_id = str(random.randint(1000, 9999))
    print(f"Generated Transaction ID: {issue_id}")
    b_id = input("Enter Book ID: ")
    m_id = input("Enter Member ID: ")
    i_date = input("Enter Issue Date (DD-MM-YYYY): ")
    file_exists = os.path.exists(CSV_FILE)
    f = open(CSV_FILE, "a", newline="")
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["Transaction_ID", "Book_ID", "Member_ID", "Issue_Date"])
    writer.writerow([issue_id, b_id, m_id, i_date])
    f.close()
    con = get_database_connection()
    if con:
        cursor = con.cursor()
        cursor.execute("UPDATE books SET status = 'Issued' WHERE book_id = %s", (b_id,))
        con.commit()
        con.close()
    print("Book transaction recorded in CSV file and book status updated successfully!")

def view_issued_csv():
    print("\n--- ISSUED BOOKS LOG (CSV FILE) ---")
    if not os.path.exists(CSV_FILE):
        print("No issue log found.")
        return
    f = open(CSV_FILE, "r")
    reader = csv.reader(f)
    for row in reader:
        if row:
            print(f"{row[0]:<18} {row[1]:<12} {row[2]:<12} {row[3]:<15}")
    f.close()

def add_library_note_text():
    print("\n--- ADD SYSTEM NOTE (TEXT FILE) ---")
    note = input("Enter notice/note for library: ")
    f = open(TEXT_FILE, "a")
    f.write(note + "\n")
    f.close()
    print("Note appended to text file successfully!")

def view_library_notes_text():
    print("\n--- LIBRARY NOTICES & NOTES (TEXT FILE) ---")
    if not os.path.exists(TEXT_FILE):
        print("No text notes file available.")
        return
    f = open(TEXT_FILE, "r")
    lines = f.readlines()
    f.close()
    if not lines:
        print("No notes available.")
    else:
        print("Notices Logged:")
        count = 1
        for line in lines:
            print(f"{count}. {line.strip()}")
            count += 1

def book_menu():
    while True:
        print("\n=============================================")
        print("           BOOK MANAGEMENT MENU (SQL)        ")
        print("=============================================")
        print("1. Add New Book")
        print("2. View All Books")
        print("3. Search Book by ID")
        print("4. Check Book Status (In Library / With Member)")
        print("5. Update Book Status Manually (Return/Issue)")
        print("6. Delete Book by ID")
        print("7. Back to Main Menu")
        print("=============================================")
        choice = input("Enter your choice (1-7): ").strip()
        if choice == "1":
            add_book_sql()
        elif choice == "2":
            display_books_sql()
        elif choice == "3":
            search_book_sql()
        elif choice == "4":
            check_book_status()
        elif choice == "5":
            update_book_status_manual()
        elif choice == "6":
            delete_book_sql()
        elif choice == "7":
            break
        else:
            print("Invalid Choice! Please enter a number from 1 to 7.")

def member_menu():
    while True:
        print("\n=============================================")
        print("       MEMBER MANAGEMENT MENU (BINARY)       ")
        print("=============================================")
        print("1. Register New Member")
        print("2. View Members List")
        print("3. Delete Member by ID")
        print("4. Back to Main Menu")
        print("=============================================")
        choice = input("Enter your choice (1-4): ").strip()
        if choice == "1":
            add_member_binary()
        elif choice == "2":
            display_members_binary()
        elif choice == "3":
            delete_member_binary()
        elif choice == "4":
            break
        else:
            print("Invalid Choice! Please enter a number from 1 to 4.")

def transaction_notes_menu():
    while True:
        print("\n=============================================")
        print("   TRANSACTIONS & SYSTEM NOTES (CSV / TEXT)  ")
        print("=============================================")
        print("1. Issue Book Record (CSV)")
        print("2. View Issued Books Log (CSV)")
        print("3. Add System Notice (Text File)")
        print("4. View Notices/Notes (Text File)")
        print("5. Back to Main Menu")
        print("=============================================")
        choice = input("Enter your choice (1-5): ").strip()
        if choice == "1":
            issue_book_csv()
        elif choice == "2":
            view_issued_csv()
        elif choice == "3":
            add_library_note_text()
        elif choice == "4":
            view_library_notes_text()
        elif choice == "5":
            break
        else:
            print("Invalid Choice! Please enter a number from 1 to 5.")

def main_menu():
    print("=============================================")
    print("        ADMIN LOGIN AUTHENTICATION           ")
    print("=============================================")
    admin_password = input("Enter Admin Password: ").strip()
    if admin_password != "admin123":
        print("Access Denied! Incorrect Admin Password.")
        return
    print("Access Granted! Welcome Admin.")
    while True:
        print("\n=============================================")
        print("          LIBRARY MANAGEMENT SYSTEM          ")
        print("=============================================")
        print("1. Book Management (SQL Database)")
        print("2. Member Management (Binary File)")
        print("3. Transactions & Notices (CSV & Text Files)")
        print("4. Exit System")
        print("=============================================")
        choice = input("Enter your choice (1-4): ").strip()
        if choice == "1":
            book_menu()
        elif choice == "2":
            member_menu()
        elif choice == "3":
            transaction_notes_menu()
        elif choice == "4":
            print("\nThank you for using the Library Management System!")
            break
        else:
            print("Invalid Choice! Please enter a number from 1 to 4.")

if __name__ == "__main__":
    main_menu()